# 🤖 Chatbot Expert AI Act (UE 2024/1689)
**Pipeline RAG : FAISS + Sentence-Transformers + Mistral 7B + LangChain**

Ce notebook :
1. Installe les dépendances
2. Parse et chunke le texte de l'AI Act
3. Construit un index vectoriel FAISS
4. Lance un chatbot RAG avec Mistral 7B via Ollama

---
## Cellule 1 — Installation des dépendances

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"])

---
## Cellule 2 — Imports

In [ ]:
import re
from pathlib import Path

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

print("Imports OK")

---
## Cellule 3 — Configuration
Modifiez les chemins si nécessaire.

In [ ]:
# --- CONFIGURATION ---
MD_FILE   = Path("C:\\Users\\Bernard\\Downloads\\RAG_project\\L-202401689FR.000101.fmx.xml.md")   # Fichier source AI Act
INDEX_DIR = Path("faiss_index")                        # Dossier de l'index FAISS
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"  # 278M params, 768 dim, 50+ langues
LLM_MODEL  = "gemma3:1b"                              # Modèle Ollama (très léger, ~1.5 Go RAM)
TOP_K      = 5                                         # Nombre de chunks récupérés par requête

print(f"Fichier source : {MD_FILE} ({'existe' if MD_FILE.exists() else 'INTROUVABLE'})")
print(f"Index FAISS    : {INDEX_DIR} ({'existe' if INDEX_DIR.exists() else 'à construire'})")
print(f"LLM            : {LLM_MODEL}")

---
## Cellule 4 — Fonctions de parsing et chunking

Le texte de l'AI Act est un règlement européen avec une structure très hiérarchisée :
- **Considérants** (1) à (180) : motivations du texte, format `|(N)|texte|`
- **13 Chapitres** > **Sections** > **113 Articles** numérotés

La stratégie de chunking exploite cette structure : **1 chunk = 1 considérant ou 1 article** (découpé par paragraphe si trop long), avec un préfixe hiérarchique et des métadonnées riches.

In [ ]:
def clean_text(text: str) -> str:
    """
    Nettoie les artefacts de la conversion Markdown depuis EUR-Lex :
    - |---|---| : séparateurs de tableau Markdown
    - |a)|texte| : listes à puces au format tableau
    - [texte](url) : liens Markdown
    - Sauts de ligne multiples
    """
    text = re.sub(r"\|---\|---\|", "", text) # Supprimer les séparateurs de tableau
    text = re.sub(r"\|([a-z]\))\|(.+?)\|", r"\1 \2", text) # Convertir les listes à puces
    text = re.sub(r"\[([^\]]+)\]\([^\)]+\)", r"\1", text) # Supprimer les liens Markdown
    text = re.sub(r"\n{3,}", "\n\n", text) # Réduire les sauts de ligne multiples à deux
    return text.strip() # Supprimer les espaces en début et fin de texte


def parse_ai_act(filepath: Path = MD_FILE) -> list[dict]:
    """
    Parse le Markdown de l'AI Act et retourne une liste de chunks.
    
    Chaque chunk = {
        "content": str,       # Texte du chunk avec préfixe hiérarchique
        "metadata": {         # Métadonnées pour le filtrage et l'affichage
            "type":           # "considerant" ou "article"
            "chapter":        # Numéro du chapitre (ex: "III")
            "chapter_title":  # Titre du chapitre
            "section":        # Numéro de section
            "article":        # Numéro d'article
            "title":          # Titre de l'article
            "paragraph":      # Numéro de paragraphe (si article découpé)
        }
    }
    """
    text = Path(filepath).read_text(encoding="utf-8")

    # Normaliser les espaces insécables (\xa0) → espaces normaux
    # Le document EUR-Lex utilise \xa0 dans "CHAPITRE I", "Article 5", etc.
    text = text.replace("\xa0", " ")

    # Supprimer le header YAML, le titre H1 et le bloc Excerpt
    text = re.sub(r"^---.*?---", "", text, count=1, flags=re.DOTALL)
    text = re.sub(r"^#\s+.*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^>\s+.*$", "", text, flags=re.MULTILINE)

    lines = text.split("\n")
    chunks = []

    # Variables de contexte hiérarchique (mises à jour au fil du parsing)
    current_chapter = ""
    current_chapter_title = ""
    current_section = ""
    current_section_title = ""

    # ========================================
    # PHASE 1 : Trouver la frontière considérants / articles
    # ========================================
    chapitre_start = None
    for idx, line in enumerate(lines):
        if line.strip() == "CHAPITRE I":
            chapitre_start = idx
            break

    i = 0  # Pointeur de ligne courant

    # ========================================
    # PHASE 2 : Parser les considérants (avant CHAPITRE I)
    # Format : |(N)|texte du considérant|
    # ========================================
    if chapitre_start:
        for idx in range(chapitre_start):
            m = re.match(r"^\|(\(\d+\))\|(.+)\|$", lines[idx])
            if not m:
                continue
            num, body = m.group(1), m.group(2)
            cleaned = clean_text(body)
            if len(cleaned) < 20:
                continue
            chunks.append({
                "content": f"Considérant {num} : {cleaned}",
                "metadata": {
                    "type": "considerant",
                    "numero": num,
                    "chapter": "",
                    "chapter_title": "",
                    "section": "",
                    "section_title": "",
                    "article": "",
                    "title": f"Considérant {num}",
                    "paragraph": "",
                },
            })
        i = chapitre_start

    # ========================================
    # PHASE 3 : Parser les articles (après CHAPITRE I)
    # On détecte 3 types de marqueurs : CHAPITRE, SECTION, Article
    # ========================================
    while i < len(lines):
        line = lines[i].strip()

        # --- Détecter un CHAPITRE (met à jour le contexte) ---
        match_chap = re.match(r"^CHAPITRE\s+([IVXLC]+)$", line)
        if match_chap:
            current_chapter = match_chap.group(1)
            current_section = ""
            current_section_title = ""
            # Le titre est sur la prochaine ligne non vide
            j = i + 1
            while j < len(lines) and lines[j].strip() == "":
                j += 1
            if j < len(lines):
                current_chapter_title = lines[j].strip()
            i = j + 1
            continue

        # --- Détecter une SECTION (met à jour le contexte) ---
        match_sec = re.match(r"^SECTION\s+(\d+)$", line)
        if match_sec:
            current_section = match_sec.group(1)
            j = i + 1
            while j < len(lines) and lines[j].strip() == "":
                j += 1
            if j < len(lines):
                current_section_title = lines[j].strip()
            i = j + 1
            continue

        # --- Détecter un Article ---
        match_art = re.match(r"^Article\s+(premier|\d+)$", line)
        if match_art:
            art_num = match_art.group(1)
            if art_num == "premier":
                art_num = "1"

            # Titre = prochaine ligne non vide
            j = i + 1
            while j < len(lines) and lines[j].strip() == "":
                j += 1
            art_title = lines[j].strip() if j < len(lines) else ""
            j += 1

            # Collecter le contenu jusqu'au prochain marqueur
            content_lines = []
            while j < len(lines):
                next_line = lines[j].strip()
                if re.match(r"^(Article\s+(premier|\d+)|CHAPITRE\s+[IVXLC]+|SECTION\s+\d+)$", next_line):
                    break
                content_lines.append(lines[j])
                j += 1

            raw_content = "\n".join(content_lines)
            cleaned = clean_text(raw_content)

            if len(cleaned) < 10:
                i = j
                continue

            # Construire le préfixe hiérarchique
            prefix_parts = []
            if current_chapter:
                prefix_parts.append(f"Chapitre {current_chapter} - {current_chapter_title}")
            if current_section:
                prefix_parts.append(f"Section {current_section} - {current_section_title}")
            prefix_parts.append(f"Article {art_num} : {art_title}")
            prefix = " > ".join(prefix_parts)

            full_content = f"{prefix}\n\n{cleaned}"

            # Si trop long → découper par paragraphe numéroté (1.  , 2.  , ...)
            if len(full_content) > 2000:
                paragraphs = re.split(r"\n(?=\d+\.\s{2,})", cleaned)
                for p_idx, para in enumerate(paragraphs):
                    para = para.strip()
                    if len(para) < 20:
                        continue
                    chunks.append({
                        "content": f"{prefix}\n\n{para}",
                        "metadata": {
                            "type": "article",
                            "chapter": current_chapter,
                            "chapter_title": current_chapter_title,
                            "section": current_section,
                            "section_title": current_section_title,
                            "article": art_num,
                            "title": art_title,
                            "paragraph": str(p_idx + 1),
                        },
                    })
            else:
                chunks.append({
                    "content": full_content,
                    "metadata": {
                        "type": "article",
                        "chapter": current_chapter,
                        "chapter_title": current_chapter_title,
                        "section": current_section,
                        "section_title": current_section_title,
                        "article": art_num,
                        "title": art_title,
                        "paragraph": "",
                    },
                })

            i = j
            continue

        i += 1

    return chunks


print("Fonctions de parsing définies ✓")

---
## Cellule 5 — Parsing du AI Act
On exécute le chunker et on affiche les statistiques.

In [ ]:
chunks = parse_ai_act()

considerants = [c for c in chunks if c["metadata"]["type"] == "considerant"]
articles     = [c for c in chunks if c["metadata"]["type"] == "article"]

print(f"Nombre total de chunks : {len(chunks)}")
print(f"  - Considérants : {len(considerants)}")
print(f"  - Articles     : {len(articles)}")
print()

# Aperçu de 3 chunks
for c in chunks[:2] + chunks[200:201]:
    print(f"[{c['metadata']['type']}] {c['metadata']['title']}")
    print(c["content"][:150] + "...")
    print()

---
## Cellule 6 — Construction de l'index FAISS

On encode les 641 chunks avec `paraphrase-multilingual-mpnet-base-v2` (278M paramètres, 768 dimensions) et on les indexe dans FAISS.

**⏱ Première exécution** : ~2-3 min (téléchargement du modèle ~1 Go + encodage).  
**Exécutions suivantes** : ~30s (modèle en cache).

Si l'index existe déjà, cette cellule le recharge simplement.

In [ ]:
# Charger le modèle d'embeddings
print(f"Chargement du modèle : {MODEL_NAME}")
embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs={"device": "cpu"},              # "cuda" si GPU disponible
    encode_kwargs={
        "normalize_embeddings": True,             # Norme L2=1 → cosine similarity = dot product
        "batch_size": 32,                         # Encode 32 textes à la fois
    },
)
print("Modèle chargé ✓")

# Construire ou recharger l'index
if INDEX_DIR.exists():
    print("Index FAISS existant trouvé → rechargement...")
    db = FAISS.load_local(str(INDEX_DIR), embeddings, allow_dangerous_deserialization=True)
else:
    print("Construction de l'index FAISS...")
    documents = [
        Document(page_content=c["content"], metadata=c["metadata"])
        for c in chunks
    ]
    db = FAISS.from_documents(documents, embeddings)
    db.save_local(str(INDEX_DIR))
    print(f"Index sauvegardé dans {INDEX_DIR}/")

print(f"Index FAISS prêt ✓ ({db.index.ntotal} vecteurs de dimension {db.index.d})")

---
## Cellule 7 — Test de la recherche vectorielle (sans LLM)
Vérifions que FAISS retrouve les bons articles avant de brancher le LLM.

In [ ]:
question_test = "Quelles sont les pratiques d'IA interdites ?"

docs = db.similarity_search(question_test, k=TOP_K)

print(f"Question : {question_test}\n")
print(f"{len(docs)} documents retrouvés :\n")
for i, doc in enumerate(docs):
    m = doc.metadata
    label = m.get("title", "")
    if m.get("article"):
        label = f"Article {m['article']} : {label}"
    if m.get("chapter"):
        label = f"Chap. {m['chapter']} > {label}"
    print(f"  {i+1}. [{m['type']}] {label}")
    print(f"     {doc.page_content[:120]}...\n")

---
## Cellule 8 — Connexion au LLM (Gemma 3 4B via Ollama)

**Prérequis** : Ollama doit tourner en local avec le modèle :
```bash
# Installer Ollama : https://ollama.com/download
ollama pull gemma3:4b
ollama serve           # si pas déjà lancé
```
Gemma 3 4B est plus léger que Mistral 7B (~3 Go vs ~5 Go de RAM) et répond beaucoup plus vite.

In [ ]:
llm = Ollama(model=LLM_MODEL, temperature=0.1, timeout=120)

# Test rapide de connexion
try:
    test = llm.invoke("Réponds juste OK.")
    print(f"Connexion Ollama OK ✓  (modèle: {LLM_MODEL})")
    print(f"Réponse test : {test[:50]}")
except Exception as e:
    print(f"ERREUR de connexion à Ollama : {e}")
    print("Vérifiez que Ollama est lancé : ollama serve")
    print(f"Et que {LLM_MODEL} est installé : ollama pull {LLM_MODEL}")

---
## Cellule 9 — Construction de la chaîne RAG

Le flux complet :
```
Question → FAISS (top-5) → Prompt (contexte + question) → Mistral 7B → Réponse
```

In [ ]:
# --- Prompt système durci (anti-hallucinations) ---
SYSTEM_PROMPT = """\
Tu es un assistant juridique spécialisé UNIQUEMENT sur le Règlement européen sur \
l'Intelligence Artificielle (AI Act, Règlement UE 2024/1689).

RÈGLES STRICTES — tu DOIS les respecter à chaque réponse :
1. Réponds UNIQUEMENT à partir du contexte ci-dessous. JAMAIS avec tes connaissances.
2. Si le contexte ne contient PAS la réponse, tu DOIS répondre exactement :
   "Cette information ne figure pas dans les extraits de l'AI Act à ma disposition."
3. Ne complète JAMAIS une réponse avec des informations extérieures au contexte.
4. Cite TOUJOURS les articles ou considérants exacts (ex: "Article 6, paragraphe 2").
5. Si la question ne concerne PAS l'AI Act, refuse poliment en disant :
   "Ma compétence se limite au Règlement UE 2024/1689 (AI Act). Je ne peux pas répondre à cette question."
6. Réponds en français, de manière structurée et concise.

Contexte (extraits officiels du Règlement UE 2024/1689) :
{context}
"""

NO_CONTEXT_RESPONSE = (
    "Je n'ai trouvé aucun passage pertinent dans l'AI Act pour répondre à cette question.\n\n"
    "Cela peut signifier que :\n"
    "- La question porte sur un sujet non couvert par le Règlement UE 2024/1689\n"
    "- La formulation de la question est trop éloignée du vocabulaire juridique du texte\n\n"
    "Essayez de reformuler votre question en utilisant des termes du règlement "
    "(ex: *système d'IA à haut risque*, *pratiques interdites*, *obligations de transparence*)."
)

SCORE_THRESHOLD = 0.4  # Seuil minimum de similarité (0-1). Augmenter = plus strict.

PROMPT = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}"),
])

# Retriever avec seuil de score : ne retourne QUE les documents au-dessus du seuil
retriever = db.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": TOP_K, "score_threshold": SCORE_THRESHOLD},
)


def format_docs(docs):
    """Concatène les documents récupérés en un seul bloc de contexte."""
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


def get_sources(docs):
    """Extrait une liste lisible des sources (pour affichage)."""
    sources = []
    for doc in docs:
        m = doc.metadata
        if m.get("type") == "article":
            label = f"Article {m['article']} : {m['title']}"
            if m.get("chapter"):
                label = f"Chapitre {m['chapter']} > {label}"
        else:
            label = m.get("title", "Considérant")
        sources.append(label)
    return sources


# Chaîne RAG complète (LangChain LCEL)
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | PROMPT
    | llm
    | StrOutputParser()
)

print("Chaîne RAG construite ✓ (avec seuil de score et prompt durci)")

---
## Cellule 10 — Fonction de question/réponse
Fonction utilitaire qui pose une question et affiche la réponse + les sources.

In [ ]:
def ask(question: str):
    """
    Pose une question au chatbot AI Act.
    Garde-fou : si le retriever ne trouve rien au-dessus du seuil,
    on refuse poliment SANS appeler le LLM (évite les hallucinations).
    """
    print(f"Question : {question}")
    print("=" * 60)

    # 1. Recherche vectorielle (avec seuil de score)
    docs = retriever.invoke(question)
    sources = get_sources(docs)

    # 2. Garde-fou : aucun document pertinent ?
    if not docs:
        print(NO_CONTEXT_RESPONSE)
        print(f"\n[Retriever : 0 documents au-dessus du seuil {SCORE_THRESHOLD}]")
        return NO_CONTEXT_RESPONSE

    # 3. Documents trouvés → générer la réponse via le LLM
    print(f"[Retriever : {len(docs)} documents trouvés]")
    response = chain.invoke(question)

    print(response)
    print("\n" + "-" * 60)
    print(f"Sources ({len(sources)}) :")
    for s in sources:
        print(f"  - {s}")
    print()
    return response

---
## Cellule 11 — Test du chatbot
Posez vos questions ici !

In [ ]:
ask("Quelles sont les pratiques d'IA interdites par l'AI Act ?")

In [ ]:
ask("Qu'est-ce qu'un système d'IA à haut risque ?")

In [ ]:
ask("Quelles sont les obligations de transparence ?")

---
## Test anti-hallucinations
Questions hors-sujet : le chatbot doit refuser poliment sans inventer de réponse.

---
## Cellule 12 — Mode interactif (boucle de chat)
Tapez vos questions en boucle. Entrez `quit` pour arrêter.

In [ ]:
ask("Qu'est-ce que le Bitcoin ?")

In [ ]:
ask("Quelle est la capitale de la France ?")

In [ ]:
print("Chatbot AI Act — Tapez 'quit' pour quitter")
print("=" * 50)

while True:
    question = input("\nVotre question : ")
    if question.strip().lower() in ("quit", "exit", "q"):
        print("Au revoir !")
        break
    if not question.strip():
        continue
    ask(question)